# 机器翻译模型（MarianMT + translation2019zh）

## 项目概览
- **模型**：MarianMT（Helsinki-NLP/opus-mt-zh-en）
- **任务**：中文 → 英文机器翻译 (Seq2Seq)
- **数据集**：translation2019zh（中英平行语料，22万条）
- **输入**：中文句子
- **输出**：生成对应的英文翻译
- **评估指标**：BLEU（SacreBLEU corpus-level）

## 与摘要任务对比
| 对比项 | 翻译（本项目） | 摘要（seq2seq_summarization） |
|--------|---------------|-------------------------------|
| 模型 | MarianMT（专用翻译模型） | mT5（多语言通用 Seq2Seq） |
| 输入/输出 | 不同语言（中→英） | 同一语言（长文→短摘要） |
| 数据量 | 220,000 条 | 百万级摘要数据 |
| 评估 | BLEU | ROUGE |

## 运行说明
- 使用快捷键 `Shift + Enter` 逐个运行 Cell
- 或点击菜单 `Cell → Run All` 一次性运行全部
- 整个流程耗时约 2-3 小时（GPU，batch_size=32）

⚠️ **数据集说明**：需要先下载 [translation2019zh](https://github.com/brightmart/nlp_chinese_corpus) 数据集，
放置在 `../../data/translation2019zh/` 目录下。

## 第一步：环境检查和库导入

In [ ]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AdamW,
    get_scheduler
)

# 任务	用哪个 AutoModel
# 文本分类	AutoModelForSequenceClassification
# 问答	AutoModelForQuestionAnswering
# 翻译 / 摘要 / T5	AutoModelForSeq2SeqLM  ← 本项目
# GPT / LLaMA 生成	AutoModelForCausalLM

from sacrebleu.metrics import BLEU
from tqdm.auto import tqdm

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("✓ 库导入成功")
print(f"✓ PyTorch 版本: {torch.__version__}")
print(f"✓ GPU 可用: {torch.cuda.is_available()}")
print(f"✓ MPS 可用: {torch.backends.mps.is_available()}")

## 第二步：数据加载

数据格式（JSON Lines）：
```json
{"chinese": "这是一句中文", "english": "This is a Chinese sentence"}
```

In [ ]:
DATA_DIR   = '../../data/translation2019zh/'
OUTPUT_DIR = './trans_model_output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(DATA_DIR, 'translation2019zh_train.json')
TEST_FILE  = os.path.join(DATA_DIR, 'translation2019zh_valid.json')

MAX_DATASET_SIZE = 220000
TRAIN_SET_SIZE   = 200000
VALID_SET_SIZE   = 20000

for name, path in [('训练/验证数据', TRAIN_FILE), ('测试数据', TEST_FILE)]:
    print(f"{name}: {path}")
    print(f"  存在: {os.path.exists(path)}")
    if os.path.exists(path):
        print(f"  大小: {os.path.getsize(path) / 1024 / 1024:.1f} MB")

print(f"\n输出目录: {OUTPUT_DIR}")

## 第三步：定义数据集类

In [ ]:
class TRANS(Dataset):
    """中英翻译数据集（JSON Lines 格式）"""

    def __init__(self, data_file, max_size=None):
        self.data = self.load_data(data_file, max_size)

    def load_data(self, data_file, max_size):
        Data = {}
        with open(data_file, 'rt', encoding='utf-8') as f:
            for idx, line in enumerate(f):
                if max_size and idx >= max_size:
                    break
                sample = json.loads(line.strip())
                Data[idx] = sample
        return Data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

print("✓ TRANS 数据集类定义成功")

## 第四步：加载分词器和数据

In [ ]:
# ============ 模型选择配置 ============
# 1. 'Helsinki-NLP/opus-mt-zh-en'   - MarianMT 中→英（推荐）⭐⭐⭐ (默认)
# 2. 'Helsinki-NLP/opus-mt-en-zh'   - MarianMT 英→中
# 3. 'google/mt5-small'             - mT5-Small（多语言通用）
#
MODEL_CHOICE = 'Helsinki-NLP/opus-mt-zh-en'  # ⬅️ 修改这里选择模型

model_name = MODEL_CHOICE
print(f"✓ 模型选择: {model_name}")

print(f"\n加载分词器: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("✓ 分词器加载成功")

print("\n加载数据集（最多 220,000 条）...")
full_data = TRANS(TRAIN_FILE, max_size=MAX_DATASET_SIZE)
train_dataset, dev_dataset = random_split(full_data, [TRAIN_SET_SIZE, VALID_SET_SIZE])
print(f"✓ 训练集: {len(train_dataset)} 条")
print(f"✓ 验证集: {len(dev_dataset)} 条")

print("\n加载测试集...")
test_dataset = TRANS(TEST_FILE)
print(f"✓ 测试集: {len(test_dataset)} 条")

## 第五步：查看数据样本

In [ ]:
for i in range(3):
    sample = train_dataset[i]
    print(f"\n=== 样本 {i+1} ===")
    print(f"中文: {sample['chinese']}")
    print(f"英文: {sample['english']}")

sample = train_dataset[0]
print("\n=== 分词示例 ===")
tokens = tokenizer(sample['chinese'], text_target=sample['english'], max_length=128, truncation=True)
print(f"input_ids 长度: {len(tokens['input_ids'])}")
print(f"labels 长度:    {len(tokens['labels'])}")
print(f"input tokens:  {tokenizer.convert_ids_to_tokens(tokens['input_ids'][:10])}")
print(f"output tokens: {tokenizer.convert_ids_to_tokens(tokens['labels'][:10])}")

## 第六步：超参数配置

In [ ]:
class Args:
    max_length       = 128
    batch_size       = 32
    num_train_epochs = 3
    learning_rate    = 1e-5
    warmup_proportion= 0.0
    weight_decay     = 0.01
    adam_beta1       = 0.9
    adam_beta2       = 0.98
    adam_epsilon     = 1e-8
    seed             = 42
    output_dir       = OUTPUT_DIR

args = Args()
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)

if torch.backends.mps.is_available():
    args.device = torch.device('mps')
elif torch.cuda.is_available():
    args.device = torch.device('cuda')
else:
    args.device = torch.device('cpu')

print("超参数配置:")
for k, v in vars(args).items():
    print(f"  {k}: {v}")

## 第七步：加载模型

In [ ]:
print(f"加载 MarianMT 模型: {model_name}...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.to(args.device)

print(f"✓ 模型加载成功")
print(f"✓ 模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"✓ 模型所在设备: {next(model.parameters()).device}")

## 第八步：创建 DataLoader

**关键**：
1. 使用 `text_target` 参数让 tokenizer 同时对源文和目标文编码，得到 `labels`
2. 用 `prepare_decoder_input_ids_from_labels` 生成解码器输入（右移一位，加 BOS）
3. 将 EOS 之后的 label 位置置为 `-100`（不计入 loss）

In [ ]:
def get_dataLoader(args, dataset, model, tokenizer, shuffle=False):
    """创建翻译任务的 DataLoader"""

    def collote_fn(batch_samples):
        batch_inputs, batch_targets = [], []
        for sample in batch_samples:
            batch_inputs.append(sample['chinese'])
            batch_targets.append(sample['english'])

        batch_data = tokenizer(
            batch_inputs,
            text_target=batch_targets,
            padding=True,
            max_length=args.max_length,
            truncation=True,
            return_tensors='pt'
        )

        # 解码器输入 = labels 右移一位（在开头加 BOS token）
        batch_data['decoder_input_ids'] = model.prepare_decoder_input_ids_from_labels(batch_data['labels'])

        # EOS 之后的位置不计入 loss
        end_token_index = torch.where(batch_data['labels'] == tokenizer.eos_token_id)[1]
        for idx, end_idx in enumerate(end_token_index):
            batch_data['labels'][idx][end_idx+1:] = -100

        return batch_data

    return DataLoader(dataset, batch_size=args.batch_size, shuffle=shuffle, collate_fn=collote_fn)


print("创建 DataLoader...")
train_dataloader = get_dataLoader(args, train_dataset, model, tokenizer, shuffle=True)
dev_dataloader   = get_dataLoader(args, dev_dataset,   model, tokenizer, shuffle=False)
test_dataloader  = get_dataLoader(args, test_dataset,  model, tokenizer, shuffle=False)

print(f"✓ 训练集批次数: {len(train_dataloader)}")
print(f"✓ 验证集批次数: {len(dev_dataloader)}")
print(f"✓ 测试集批次数: {len(test_dataloader)}")

## 第九步：设置优化器和学习率调度器

In [ ]:
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': args.weight_decay},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0}
]

t_total = len(train_dataloader) * args.num_train_epochs
args.warmup_steps = int(t_total * args.warmup_proportion)

optimizer = AdamW(
    optimizer_grouped_parameters,
    lr=args.learning_rate,
    betas=(args.adam_beta1, args.adam_beta2),
    eps=args.adam_epsilon
)
lr_scheduler = get_scheduler(
    'linear', optimizer,
    num_warmup_steps=args.warmup_steps,
    num_training_steps=t_total
)

print(f"优化器: AdamW，学习率: {args.learning_rate}")
print(f"调度器: Linear Warmup，总步数: {t_total}")

## 第十步：定义训练和评估函数

In [ ]:
def train_loop(args, dataloader, model, optimizer, lr_scheduler, epoch, total_loss):
    """训练一个 epoch"""
    model.train()
    finish_step_num = epoch * len(dataloader)
    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch+1} Training')

    for step, batch_data in enumerate(progress_bar, start=1):
        batch_data = batch_data.to(args.device)
        outputs    = model(**batch_data)
        loss       = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'avg_loss': f'{total_loss / (finish_step_num + step):.4f}'})

    return total_loss


def test_loop(args, dataloader, model, tokenizer):
    """
    评估：生成翻译并计算语料级 BLEU（SacreBLEU）
    corpus-level BLEU 比 sentence-level 更稳定可靠。
    """
    bleu_metric = BLEU()
    preds, labels = [], []

    model.eval()
    with torch.no_grad():
        for batch_data in tqdm(dataloader, desc='Evaluating'):
            batch_data = batch_data.to(args.device)
            generated_tokens = model.generate(
                batch_data['input_ids'],
                attention_mask=batch_data['attention_mask'],
                max_length=args.max_length
            ).cpu().numpy()

            label_tokens = batch_data['labels'].cpu().numpy()
            label_tokens = np.where(label_tokens != -100, label_tokens, tokenizer.pad_token_id)

            decoded_preds  = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
            decoded_labels = tokenizer.batch_decode(label_tokens, skip_special_tokens=True)

            preds  += [p.strip() for p in decoded_preds]
            labels += [[l.strip()] for l in decoded_labels]

    return bleu_metric.corpus_score(preds, labels).score

print("✓ train_loop / test_loop 定义成功")

## 第十一步：开始训练 ⭐⭐⭐

⚠️ **这一步需要较长时间**（约 2-3 小时，GPU batch=32）

In [ ]:
training_history = {'train_loss': [], 'dev_bleu': []}
best_bleu        = 0.0
best_model_path  = os.path.join(OUTPUT_DIR, 'best_model.bin')
total_loss       = 0.0

print("开始训练...\n")
for epoch in range(args.num_train_epochs):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{args.num_train_epochs}")
    print(f"{'='*60}")

    total_loss = train_loop(args, train_dataloader, model, optimizer, lr_scheduler, epoch, total_loss)
    avg_loss   = total_loss / ((epoch + 1) * len(train_dataloader))
    print(f"\n平均训练损失: {avg_loss:.4f}")
    training_history['train_loss'].append(avg_loss)

    dev_bleu = test_loop(args, dev_dataloader, model, tokenizer)
    print(f"验证集 BLEU: {dev_bleu:.4f}")
    training_history['dev_bleu'].append(dev_bleu)

    if dev_bleu > best_bleu:
        best_bleu = dev_bleu
        torch.save(model.state_dict(), best_model_path)
        print(f"✓ 保存最佳模型 (BLEU: {best_bleu:.4f})")

print("\n" + "="*60)
print("✅ 训练完成！")
print("="*60)

## 第十二步：绘制训练曲线

In [ ]:
epochs = range(1, args.num_train_epochs + 1)
fig, (ax_loss, ax_bleu) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('模型训练收敛曲线', fontsize=15, fontweight='bold')

ax_loss.plot(epochs, training_history['train_loss'], marker='o', linewidth=2, markersize=8)
ax_loss.set_xlabel('Epoch', fontsize=12)
ax_loss.set_ylabel('Loss', fontsize=12)
ax_loss.set_title('训练 Loss 曲线', fontsize=13, fontweight='bold')
ax_loss.set_xticks(epochs)
ax_loss.grid(True, alpha=0.3)

ax_bleu.plot(epochs, training_history['dev_bleu'], marker='s', linewidth=2, markersize=8, color='orange')
ax_bleu.set_xlabel('Epoch', fontsize=12)
ax_bleu.set_ylabel('BLEU', fontsize=12)
ax_bleu.set_title('验证集 BLEU 曲线', fontsize=13, fontweight='bold')
ax_bleu.set_xticks(epochs)
ax_bleu.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ 曲线已保存")

## 第十三步：加载最佳模型和预测函数

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=args.device))
model.eval()
print(f"✓ 加载最佳模型: {best_model_path}")


def translate(sentence, model, tokenizer, device, max_length=128):
    """翻译单个中文句子为英文"""
    inputs = tokenizer(
        sentence,
        max_length=max_length,
        truncation=True,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        generated_tokens = model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_length
        ).cpu().numpy()

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

print("✓ translate 函数定义成功")

## 第十四步：测试集预测示例

In [ ]:
print("\n" + "="*80)
print("测试集预测示例（前 5 条）")
print("="*80)

for i in range(min(5, len(test_dataset))):
    sample     = test_dataset[i]
    prediction = translate(sample['chinese'], model, tokenizer, args.device)
    print(f"\n示例 {i+1}:")
    print(f"🇨🇳 中文: {sample['chinese']}")
    print(f"✓ 参考:  {sample['english']}")
    print(f"🤖 翻译: {prediction}")
    print("-" * 80)

## 第十五步：自定义输入预测

In [ ]:
custom_sentences = [
    "人工智能正在改变世界。",
    "北京是中国的首都，也是一座有着悠久历史的城市。",
    "深度学习在图像识别和自然语言处理方面取得了显著进展。",
    "今天天气很好，适合户外活动。",
]

print("\n" + "="*80)
print("自定义输入翻译")
print("="*80)

for idx, sentence in enumerate(custom_sentences):
    translation = translate(sentence, model, tokenizer, args.device)
    print(f"\n示例 {idx+1}:")
    print(f"🇨🇳 中文: {sentence}")
    print(f"🤖 英文: {translation}")

## 第十六步：测试集 BLEU 最终评估

In [ ]:
print("在测试集上计算 BLEU...")
test_bleu = test_loop(args, test_dataloader, model, tokenizer)
print(f"\n✓ 测试集 BLEU: {test_bleu:.4f}")

## 第十七步：模型保存和总结

In [ ]:
model_save_path = os.path.join(OUTPUT_DIR, 'final_model')
os.makedirs(model_save_path, exist_ok=True)
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"✓ 模型已保存到: {model_save_path}")

with open(os.path.join(OUTPUT_DIR, 'training_history.json'), 'w', encoding='utf-8') as f:
    json.dump(training_history, f, indent=2)

print("\n" + "="*80)
print("✅ 项目总结")
print("="*80)
print(f"""
【模型信息】
  架构: MarianMT (Helsinki-NLP/opus-mt-zh-en)
  任务: 中文 → 英文机器翻译
  数据集: translation2019zh ({TRAIN_SET_SIZE} 训练 / {VALID_SET_SIZE} 验证)
  设备: {args.device}

【最终指标】
  验证集最佳 BLEU: {best_bleu:.4f}
  测试集 BLEU:    {test_bleu:.4f}

【参考结果】
  经过 3 轮训练，Marian 模型在测试集的 BLEU 约为 52.96（V100, batch=32）

【下一步】
  1. 加载保存的模型:
     from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
     tokenizer = AutoTokenizer.from_pretrained('{model_save_path}')
     model = AutoModelForSeq2SeqLM.from_pretrained('{model_save_path}')
  2. 增大 batch_size 或数据量以提升 BLEU
  3. 调整 beam search 参数（num_beams=4/8）提升翻译质量
""")
print("="*80)